# Phase 15 — AW seed=13 reproducibility on Kaggle GPU
Re-runs `scripts/11f_train_finbert_agreement_weighted.py --seed 13 --weight_schedule linear --num_epochs 3 --batch_size 16` on a T4 to confirm the Phase 11 envelope `0.8863 +/- 0.0023`.

In [ ]:
!pip install -q transformers==4.49.0 datasets==3.* accelerate==1.* scikit-learn==1.5.* pandas numpy

In [ ]:
import os, sys, json, subprocess, glob, datetime, pathlib, shutil
INPUT = pathlib.Path('/kaggle/input')
WORK  = pathlib.Path('/kaggle/working')
REPO  = WORK / 'cara_finsent'
REPO.mkdir(parents=True, exist_ok=True)
(REPO / 'data/processed/gold').mkdir(parents=True, exist_ok=True)
(REPO / 'scripts').mkdir(parents=True, exist_ok=True)
(REPO / 'src/cara_finsent').mkdir(parents=True, exist_ok=True)
for src in INPUT.rglob('latest_gold_*.csv'):
    shutil.copy2(src, REPO / 'data/processed/gold' / src.name)
for src in INPUT.rglob('11f_train_finbert_agreement_weighted.py'):
    shutil.copy2(src, REPO / 'scripts' / src.name)
for src in INPUT.rglob('cara_finsent/*.py'):
    shutil.copy2(src, REPO / 'src/cara_finsent' / src.name)
print(sorted((REPO / 'data/processed/gold').iterdir()))
print(sorted((REPO / 'scripts').iterdir()))

In [ ]:
os.chdir(REPO)
import torch; print('cuda?', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
import sys; sys.path.insert(0, str(REPO / 'src'))
cmd = ['python', 'scripts/11f_train_finbert_agreement_weighted.py',
       '--seed', '13', '--weight_schedule', 'linear',
       '--num_epochs', '3', '--batch_size', '16']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd, glob, json, datetime
summaries = sorted(glob.glob('results/**/finbert_agreement_weighted_*_summary*.csv', recursive=True))
print(summaries[-3:])
if summaries:
    df = pd.read_csv(summaries[-1])
    print(df.to_string(index=False))
    ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    out = WORK / f'phase15_aw_gpu_repro_{ts}.csv'
    df.to_csv(out, index=False)
    print('OUT', out)
    macro = df.filter(like='macro_f1').iloc[0,0] if df.filter(like='macro_f1').shape[1] else None
    print('PASS?' , macro is not None and macro >= 0.8817, 'macro_f1=', macro)